# Week 3 Lab：Gaussian diffusion 與解析 score

## 學習目標
- 在 2D Gaussian mixture 上實作 forward noising。
- 由 mixture responsibility 精確算出 $\nabla_x\log p_t(x)$。
- 用 Tweedie 關係把 score 轉回 posterior mean。

> **誠實註記**：這是可解析的 mixture toy，不是訓練完成的 diffusion checkpoint；所有 score 都由已知密度直接計算。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 31415
rng = np.random.default_rng(SEED)
centers = np.array([[-1.6, -0.8], [-1.2, 1.1], [1.3, -1.0], [1.5, 1.0]])
data_std = 0.18

def sample_data(n):
    labels = rng.integers(0, len(centers), size=n)
    return centers[labels] + data_std * rng.standard_normal((n, 2))

def alpha_sigma(t):
    return np.cos(0.5 * np.pi * t), np.sin(0.5 * np.pi * t)

x0 = sample_data(1500)
eps = rng.standard_normal(x0.shape)
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, t in zip(axes, [0.0, 0.3, 0.6, 0.9]):
    a, s = alpha_sigma(t)
    xt = a * x0 + s * eps
    ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.35)
    ax.set(title=f't = {t:.1f}', xlim=(-3, 3), ylim=(-3, 3), aspect='equal')
plt.show()

In [ ]:
def mixture_terms(x, t):
    x = np.atleast_2d(x)
    a, s = alpha_sigma(t)
    means = a * centers
    var = (a * data_std) ** 2 + s ** 2
    logp = -0.5 * np.sum((x[:, None, :] - means[None, :, :]) ** 2, axis=2) / var
    logp -= np.max(logp, axis=1, keepdims=True)
    weight = np.exp(logp)
    responsibility = weight / weight.sum(axis=1, keepdims=True)
    return means, var, responsibility

def exact_score(x, t):
    x = np.atleast_2d(x)
    means, var, responsibility = mixture_terms(x, t)
    component_scores = (means[None, :, :] - x[:, None, :]) / var
    return np.sum(responsibility[:, :, None] * component_scores, axis=1)

def exact_posterior_mean(x, t):
    a, s = alpha_sigma(t)
    return (np.atleast_2d(x) + s ** 2 * exact_score(x, t)) / a

probe = np.array([[0.2, -0.4], [1.0, 0.8]])
print('exact score at t=.55:\n', np.round(exact_score(probe, 0.55), 3))

In [ ]:
grid = np.linspace(-3, 3, 25)
gx, gy = np.meshgrid(grid, grid)
points = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, t in zip(axes, [0.2, 0.5, 0.8]):
    score = exact_score(points, t)
    speed = np.linalg.norm(score, axis=1)
    ax.quiver(points[:, 0], points[:, 1], score[:, 0], score[:, 1], speed,
              cmap='viridis', angles='xy', scale=45, width=0.004)
    ax.set(title=f'exact score, t={t:.1f}', xlim=(-3, 3), ylim=(-3, 3), aspect='equal')
plt.show()

In [ ]:
t = 0.62
clean = sample_data(600)
a, s = alpha_sigma(t)
noisy = a * clean + s * rng.standard_normal(clean.shape)
posterior = exact_posterior_mean(noisy, t)
print('single-sample MSE before Tweedie:', np.mean((noisy / a - clean) ** 2))
print('single-sample MSE after  Tweedie:', np.mean((posterior - clean) ** 2))
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(noisy[:, 0], noisy[:, 1], s=6, alpha=.2, label='$x_t$')
ax.scatter(posterior[:, 0], posterior[:, 1], s=7, alpha=.45, label='$E[x_0|x_t]$')
ax.scatter(centers[:, 0], centers[:, 1], marker='x', s=90, label='mixture centers')
ax.set(aspect='equal', title='Tweedie posterior mean is a denoiser')
ax.legend()
plt.show()

## 讀者練習 / TODO
把 `data_std` 改成 `0.45` 後重跑：比較 mode 之間的 score 分界與 posterior mean。請解釋為什麼重疊越大，單一 noisy sample 的來源越不確定。